In [ ]:
!pip install requests
!pip install xmltodict
!pip install pycatastro
!pip install geopandas folium
!pip install feedparser

download_and_unzip función:

Esta función toma dos argumentos: url (la URL del archivo ZIP a descargar) y output_folder (la carpeta donde se almacenará el archivo descargado y descomprimido).
Intenta descargar el archivo ZIP desde la URL proporcionada y luego descomprimirlo en la carpeta de salida.
Si se produce algún error durante el proceso de descarga o descompresión, se captura la excepción y se imprime un mensaje de error.


In [49]:
import os
import shutil
import requests
import zipfile
import feedparser

def download_and_unzip(url, output_folder):
    file_name = os.path.join(output_folder, os.path.basename(url))
    try:
        # Descargar el archivo
        with requests.get(url, stream=True) as response:
            response.raise_for_status()  # Lanza una excepción si la descarga no fue exitosa
            with open(file_name, 'wb') as file:
                shutil.copyfileobj(response.raw, file)
        print(f"Archivo descargado: {file_name}")

        # Descomprimir el archivo ZIP
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall(output_folder)
        print(f"Archivo descomprimido en: {output_folder}")
    except Exception as e:
        print(f"Error al descargar y descomprimir {url}: {e}")

"""
Esta es la función principal del programa.
Solicita al usuario que ingrese la carpeta de salida donde se guardarán los archivos descargados y descomprimidos.
Verifica si se proporcionó una carpeta de salida. Si no se proporciona, muestra un mensaje de error y sale de la función.
Crea la carpeta de salida si no existe utilizando os.makedirs.
"""

### El siguiente Script Descarga todo Asturias


Define la URL del feed ATOM que contiene enlaces a archivos ZIP de parcelas catastrales para Asturias.
Utiliza feedparser para analizar el feed ATOM y obtener una lista de entradas que representan los archivos ZIP a descargar.
Itera sobre las entradas del feed y para cada entrada, obtiene la URL del archivo ZIP y llama a la función download_and_unzip para descargar y descomprimir el archivo en la carpeta de salida.

"""
Define la URL del feed ATOM que contiene enlaces a archivos ZIP de parcelas catastrales para Asturias.
Utiliza feedparser para analizar el feed ATOM y obtener una lista de entradas que representan los archivos ZIP a descargar.
Itera sobre las entradas del feed y para cada entrada, obtiene la URL del archivo ZIP y llama a la función download_and_unzip para descargar y descomprimir el archivo en la carpeta de salida.
"""


In [ ]:


def main_download():
    output_folder = input("Introduce la carpeta de salida: ")

    if not output_folder:
        print("Debes proporcionar la carpeta de salida.")
        return

    os.makedirs(output_folder, exist_ok=True)  # Crea la carpeta de salida si no existe

    # Define la URL del feed ATOM para Asturias
    atom_url = 'http://www.catastro.minhap.es/INSPIRE/CadastralParcels/33/ES.SDGC.CP.atom_33.xml'

    # Analizar el feed ATOM
    feed = feedparser.parse(atom_url)

    # Iterar sobre las entradas del feed
    for entry in feed.entries:
        # Obtener la URL del archivo ZIP del conjunto de datos de parcelas catastrales
        url = entry.links[0].href
        download_and_unzip(url, output_folder)

    print("Descarga y descompresión completadas.")

# Ejecuta la función principal de descarga
main_download()

 Con este modelo se descarga directamente un .zip en el navegador
 http://www.catastro.minhap.es/INSPIRE/CadastralParcels/33/33008-CABRALES/A.ES.SDGC.CP.33008.zip

In [50]:
import os
import csv
import shutil
import pandas as pd

# Ruta de la carpeta donde se encuentran los archivos
ruta_Asturias = '/content/sample_data/Asturias'
# Crear la carpeta si no existe
if not os.path.exists(ruta_Asturias):
    os.makedirs(ruta_Asturias)

# Ruta del archivo CSV con los códigos postales y nombres de municipios
ruta_csv = '/content/drive/MyDrive/Catastro/BD_Catastro_Actual/CP.csv'

# Ruta de la nueva carpeta madre "Parcelas-Asturias"
ruta_soloparcelas = '/content/sample_data/Parcelas-Asturias'

# Crear la carpeta si no existe
if not os.path.exists(ruta_soloparcelas):
    os.makedirs(ruta_soloparcelas)


In [51]:
# Obtener una lista de todos los archivos con extensión ".cadastralparcel.gml"
archivos_soloparcelas = [archivo for archivo in os.listdir(ruta_Asturias)
  if archivo.endswith('cadastralparcel.gml')]

Filtramos los archivos que acaban con una extensión determinada en este caso cadastralparcel.gml

In [ ]:
import pandas as pd

# Ruta del archivo CSV con los códigos postales y nombres de municipios
ruta_csv = '/content/drive/MyDrive/Catastro/BD_Catastro_Actual/CP.csv'

# Cargar el archivo CSV en un DataFrame
df_cp = pd.read_csv(ruta_csv, encoding='latin-1',sep=';')


# Mostrar las primeras líneas del DataFrame de manera atractiva
from IPython.display import display
display(df_cp.head())

In [53]:
import re
# Lista para almacenar los códigos postales
codigos_postales = []

# Iterar sobre los archivos filtrados
for archivo in archivos_soloparcelas:
    # Extraer el código postal del nombre del archivo
    codigo_postal_match = re.search(r'\d+', archivo)
    if codigo_postal_match:
        codigo_postal = codigo_postal_match.group()
        codigos_postales.append(codigo_postal)  # Agregar el código postal a la lista
    else:
        print(f"No se pudo extraer el código postal del archivo {archivo}")
        continue


In [54]:
# Iterar sobre los archivos filtrados
for archivo in archivos_soloparcelas:
    # Extraer el código postal del nombre del archivo
    codigo_postal_match = re.search(r'\d+', archivo)
    if codigo_postal_match:
        codigo_postal = codigo_postal_match.group()
        codigos_postales.append(codigo_postal)  # Agregar el código postal a la lista
    else:
        print(f"No se pudo extraer el código postal del archivo {archivo}")
        continue

    # Obtener una lista de nombres de municipios correspondientes al código postal
    nombres_municipios = df_cp[df_cp['codigo_postal'] == int(codigo_postal)]['nombre_municipio'].tolist()

    # Nuevo nombre del archivo con nombres de municipios separados por guiones
    nuevo_nombre = f"Parcelas-{codigo_postal}-{'-'.join(nombres_municipios)}.gml"

   # Ruta completa del archivo original
    ruta_original = os.path.join(ruta_archivos, archivo)

    # Ruta completa del nuevo archivo en la nueva carpeta
    ruta_nuevo = os.path.join(ruta_nueva_carpeta, nuevo_nombre)

    # Mover y renombrar el archivo a la nueva carpeta
    shutil.copy(ruta_original, ruta_nuevo)

print("Proceso completado.")

Proceso completado.
